# 623 Stride v19: chronological global/local action-grammar LSTM

The external input archive is reused byte-for-byte from v18: only PC and current address enter the model. v19 trains the contract-defined h8 and h16 points with a chronological global LSTM, an exact-PC local LSTM, causal same-PC delta/reuse-age features, and a learned soft validity gate. A sampled rank-wise STOP/EMIT grammar learns request count; exact signed increments use canonical ZigZag + LEB128. A teacher-prefix loss branch is isolated from the hard sampled main rollout. Stateless keyed inverse-CDF sampling gives common random numbers; nontermination fails closed without truncation. There is no threshold, hurdle, Poisson, GMM, budget, degree cap, candidate input, private Stride state, page rule, or future row.

In [ ]:
import hashlib, math, os, pathlib, shutil, subprocess, sys, tarfile, torch
from google.colab import userdata
assert torch.cuda.is_available(), 'Select a GPU runtime (A100 preferred)'
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
REPO='/content/cache_arch'; TOKEN=userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add GITHUB_TOKEN to Colab Secrets'
ASKPASS='/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n'); os.chmod(ASKPASS,0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':ASKPASS,'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN':TOKEN})
try:
    if not os.path.isdir(REPO): subprocess.run(['git','clone','https://github.com/Angelawoo572/cache_arch.git',REPO],check=True,env=env)
    else: subprocess.run(['git','-C',REPO,'pull','--ff-only','origin','main'],check=True,env=env)
finally: pathlib.Path(ASKPASS).unlink(missing_ok=True)
print(torch.cuda.get_device_name(0),subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())

In [ ]:
from google.colab import drive, files
import json
drive.mount('/content/drive')
SCRIPT=f'{REPO}/formal_NN_training/experiments/623_offline_lstm_stride/python/train_and_offline_infer.py'
MODEL_CONTRACT=json.loads(subprocess.check_output([sys.executable,SCRIPT,'--describe-model-points'],text=True))
RUN_ID=MODEL_CONTRACT['run_id']; DRIVE_ROOT=f'/content/drive/MyDrive/cache_prefetch_623_stride/{RUN_ID}'
INPUT_DIR=f'{DRIVE_ROOT}/colab_input'; OUTPUT_ROOT=f'{DRIVE_ROOT}/colab_output'; os.makedirs(DRIVE_ROOT,exist_ok=True)
name=f'{RUN_ID}.colab_input.tar.gz'; uploaded=files.upload(); assert name in uploaded,f'Select {name}'
archive=f'{DRIVE_ROOT}/{name}'; pathlib.Path(archive).write_bytes(uploaded[name])
if os.path.isdir(INPUT_DIR): shutil.rmtree(INPUT_DIR)
os.makedirs(INPUT_DIR,exist_ok=True)
with tarfile.open(archive,'r:gz') as handle: handle.extractall(INPUT_DIR)
for record in pathlib.Path(f'{INPUT_DIR}/SHA256SUMS').read_text().splitlines():
    expected,item=record.split(maxsplit=1); item=item.lstrip('*')
    assert hashlib.sha256(pathlib.Path(f'{INPUT_DIR}/{item}').read_bytes()).hexdigest()==expected
print('verified',archive)

In [ ]:
import gzip, json
TRACE=MODEL_CONTRACT['trace']; POLICY=MODEL_CONTRACT['policy']; ROLES=('train','guard','eval')
INPUTS={role:{'stream':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_stream.csv.gz','candidates':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_candidates.csv.gz'} for role in ROLES}
for role,items in INPUTS.items():
    for path in items.values(): assert os.path.isfile(path),path
manifest=json.loads(pathlib.Path(f'{INPUT_DIR}/collection_manifest.json').read_text())
expected={'status':'PASS','experiment_revision':MODEL_CONTRACT['experiment_revision'],'neural_role':'standalone_direct_action_prefetcher','source_decision_effective_external_input':['pc','addr'],'same_external_input_contract':True,'training_inference_input_encoder_identical':True,'decoder_training_mode':'free_running_autoregressive_same_as_inference','decoder_previous_teacher_action_used_as_input':False,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True,'normal_policy_request_rate_used_as_budget':False,'normal_policy_constants_used_by_neural_inference':False,'probability_threshold_used':False,'neural_degree_cap':None,'fixed_page_offset_classes':None,'same_page_rule_used_by_neural_inference':False,'future_label_window_used':False,'inference_policy_hardcodes_used':False,'nn_generates_own_target_addresses':True}
bad={k:(manifest.get(k),v) for k,v in expected.items() if manifest.get(k)!=v}; assert not bad,bad
assert manifest['training_runtime_fields']==['pc','addr']==manifest['inference_runtime_fields']
assert manifest['experiment_revision']==MODEL_CONTRACT['experiment_revision']

In [ ]:
LOCAL_OUTPUT=f'/content/{RUN_ID}_colab_output'
if os.path.isdir(LOCAL_OUTPUT): shutil.rmtree(LOCAL_OUTPUT)
os.makedirs(LOCAL_OUTPUT)
SPECS=[{'tag':point['model_tag'],'family':point['model_family'],'size':point['model_size'],'pair':point['architecture_pair_id'],'parameters':point['parameter_count']} for point in MODEL_CONTRACT['points']]
SWEEP=[]
for spec in SPECS:
 out=f"{LOCAL_OUTPUT}/{spec['tag']}"; cmd=[sys.executable,SCRIPT,'--policy',POLICY]
 for role in ROLES: cmd += [f'--{role}-stream',INPUTS[role]['stream'],f'--{role}-candidates',INPUTS[role]['candidates']]
 cmd += ['--out-dir',out,'--model-family',spec['family'],'--model-size',str(spec['size']),'--pair-id',spec['pair'],'--device','cuda','--seed','7','--decoder-seed','7','--epochs','10','--chunk-len','1024','--accumulate-chunks','16']
 print('\nTraining',spec['tag'],' '.join(cmd),flush=True); subprocess.run(cmd,check=True)
 meta=json.loads(pathlib.Path(f'{out}/run_metadata.json').read_text())
 expected={'model_tag':spec['tag'],'model_family':'lstm','track_model_family':'lstm','model_revision':MODEL_CONTRACT['model_revision'],'parameter_count':spec['parameters'],'runtime_feature_count':MODEL_CONTRACT['runtime_feature_count'],'raw_runtime_feature_count':MODEL_CONTRACT['raw_runtime_feature_count'],'pc_local_runtime_feature_count':MODEL_CONTRACT['pc_local_runtime_feature_count'],'matched_normal_prefetcher':POLICY,'same_external_input_contract':True,'training_inference_input_encoder_identical':True,'decoder_training_mode':MODEL_CONTRACT['decoder_training_mode'],'decoder_previous_teacher_action_used_as_input':True,'decoder_previous_teacher_action_input_scope':'isolated_loss_only_teacher_prefix_likelihood_branch','decoder_previous_teacher_action_used_as_main_rollout_input':False,'teacher_prefix_tokens_condition_loss_logits':True,'teacher_prefix_tokens_recurrently_advance_loss_branch_state':True,'teacher_prefix_tokens_mutate_main_rollout_state':False,'decoder_free_running_self_test':'PASS','normal_policy_outputs_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True,'probability_threshold_used':False,'threshold_related_hardcodes_used':False,'neural_degree_cap':None,'derived_features_use_teacher_or_future':False,'learned_local_validity_gate':True,'gate_training_objective':'NOT_APPLICABLE_no_separate_hurdle_gate','gate_decoding_rule':'NOT_APPLICABLE_STOP_EMIT_is_action_token','request_count_training_objective':MODEL_CONTRACT['request_count_training_objective'],'request_count_decoding_rule':'stateless_event_rank_keyed_categorical_inverse_cdf_until_STOP','poisson_objective_used':False,'poisson_decoder_used':False,'gmm_objective_used':False,'gmm_decoder_used':False,'delta_mixture_components':0,'delta_training_objective':MODEL_CONTRACT['delta_training_objective'],'delta_decoding_rule':'stateless_keyed_inverse_cdf_exact_ZigZag_LEB128_signed_increment','delta_decoder_feedback_rule':'main_rollout_uses_only_actual_hard_sampled_STOP_EMIT_payload_bits_continuation_tokens','delta_codec':'signed_ZigZag_then_canonical_LEB128','delta_codec_max_bytes':MODEL_CONTRACT['leb128_max_bytes'],'deterministic_decoding':False,'stochastic_decoding':True,'stochastic_decoding_reproducible':True,'common_random_numbers_across_capacities':True,'strict_common_random_numbers_across_capacities':True,'cross_event_rng_state_used':False,'cross_event_probability_credit_used':False,'sampled_outputs_used_as_decoder_feedback':True,'decoder_probability_mass_carries_train_guard_history':False,'decoder_sampling_roles':['train','eval'],'decoder_train_sampling_performed':True,'event_keyed_crn_self_test':'PASS','rankwise_stop_emit_self_test':'PASS','zigzag_leb128_exact_codec_self_test':'PASS','main_rollout_isolation_self_test':'PASS','teacher_prefix_loss_isolation_self_test':'PASS','stop_sampler_representability_self_test':'PASS','always_emit_nontermination_watchdog_self_test':'PASS','guard_role':'causal_input_history_warmup_and_audit_only','experiment_revision':MODEL_CONTRACT['experiment_revision']}
 bad={k:(meta.get(k),v) for k,v in expected.items() if meta.get(k)!=v}; assert not bad,bad
 assert meta['training_runtime_fields']==['pc','addr']==meta['inference_runtime_fields']
 stats=meta['request_count_training_label_statistics']; assert stats['decision_callbacks']==stats['positive_callbacks']+stats['zero_callbacks'] and stats['positive_callbacks']>0 and stats['zero_callbacks']>0,stats
 key_fields=['sampler_revision','decoder_seed','trace','policy','role','epoch','event_index','action_rank','field','codec_position']; sampler=meta['decoder_sampler']; assert sampler['sampler_revision']=='splitmix64_event_rank_field_inverse_cdf_crn_v2' and sampler['key_fields']==key_fields and sampler['categorical_method']=='inverse_cdf' and sampler['cross_event_rng_state'] is False,sampler
 encoder_hashes={meta.get('runtime_encoder_sha256'),meta.get('training_runtime_encoder_sha256'),meta.get('inference_runtime_encoder_sha256')}; assert len(encoder_hashes)==1 and isinstance(next(iter(encoder_hashes)),str) and len(next(iter(encoder_hashes)))==64,encoder_hashes
 assert meta.get('fail_closed_nontermination_watchdog_ranks')==MODEL_CONTRACT['nontermination_watchdog_ranks'] and meta.get('nontermination_watchdog_is_policy_degree_cap') is False and meta.get('sampler_minimum_open_midpoint_uniform')==MODEL_CONTRACT['sampler_min_uniform']
 SWEEP.append({k:meta[k] for k in ('model_tag','model_family','model_size','architecture_pair_id','parameter_count','decision_rule','offline_normal_entries','offline_nn_entries','heldout_behavior_metrics')})
if os.path.isdir(OUTPUT_ROOT): shutil.rmtree(OUTPUT_ROOT)
shutil.copytree(LOCAL_OUTPUT,OUTPUT_ROOT)
pathlib.Path(f'{OUTPUT_ROOT}/sweep_manifest.json').write_text(json.dumps({'trace':TRACE,'input_revision':MODEL_CONTRACT['experiment_revision'],'model_revision':MODEL_CONTRACT['model_revision'],'points':SWEEP},indent=2)+'\n')
print(json.dumps(SWEEP,indent=2))

In [ ]:
OUTPUT_ARCHIVE=f'{DRIVE_ROOT}/{RUN_ID}.colab_output.tar.gz'
with tarfile.open(OUTPUT_ARCHIVE,'w:gz') as archive:
 for item in pathlib.Path(OUTPUT_ROOT).iterdir(): archive.add(item,arcname=item.name)
print('DONE',OUTPUT_ARCHIVE,os.path.getsize(OUTPUT_ARCHIVE),'bytes')
files.download(OUTPUT_ARCHIVE)

Copy the v19 output archive to the matching server run and launch replay. The server requires the chronological global/local encoder, learned soft validity, sampled STOP/EMIT grammar, exact ZigZag/LEB128 increments, keyed CRN evidence, hard self-feedback, and matching input hashes; it rejects thresholds, Poisson/GMM decoders, budgets, and degree caps.